In [2]:
import os

In [3]:
%pwd

'c:\\Users\\palla\\OneDrive\\Desktop\\MLFLOW\\datascienceproject\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\palla\\OneDrive\\Desktop\\MLFLOW\\datascienceproject'

In [7]:
from dataclasses import dataclass
from pathlib import Path

#  normal class we need to use self to access the variables but in dataclass we don't need to use self to access the variables
@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [10]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories


In [21]:
class ConfiguartionManager:
    def __init__(self, 
                config_filepath= CONFIG_FILE_PATH,
                params_filepath= PARAMS_FILE_PATH,
                schema_filepath= ScHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        # this is according to the config.yaml file structure
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir= config.root_dir,
            source_URL= config.source_URL,
            local_data_file= config.local_data_file,  
            unzip_dir= config.unzip_dir
        )
        return data_ingestion_config

In [22]:
import os
import urllib.request as request
from src.datascience import logger
import zipfile # to extract the zip file

In [23]:
# component DataIngestion
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
# Downloading the zip file from the source URL and saving it to the local data file path specified in the configuration. If the file already exists, it logs that information instead of downloading it again.
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url= self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"FIle already exists")

    def extract_zip_file(self):
        """
        zip_file_path:str
        Extracts the zip file into data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok= True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
              zip_ref.extractall(unzip_path
        )

In [24]:
try:
    config = ConfiguartionManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-09-20 10:53:36,335]: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-20 10:53:36,337]: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-20 10:53:36,339]: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-20 10:53:36,341]: INFO: common: created directory at: artifacts]
[2026-09-20 10:53:36,343]: INFO: common: created directory at: artifacts/data_ingestion]
[2026-09-20 10:53:37,925]: INFO: 2446224622: artifacts/data_ingestion/data.zip downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 8C9A:44346:1F9174:5385BD:6AAF6DD8
x-github-edge-region: centr